In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, RadioButtons, VBox, HBox, HTML, Output, Layout
from IPython.display import display, clear_output

# ============================================================
# DITHER AND QUANTIZATION ERROR
# ============================================================

# ============================================================
# CSS
# ============================================================

display(HTML("""
<style>

.dither-radio .widget-radio-box {
    display: flex !important;
    flex-direction: row !important;
    flex-wrap: nowrap !important;
    gap: 22px !important;
    align-items: center !important;
}

.dither-radio .widget-radio-box label {
    margin: 0 !important;
    white-space: nowrap !important;
}

.dither-radio > label {
    display: none !important;
}

</style>
"""))

# ============================================================
# SHORT DOCUMENTATION
# ============================================================

documentation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:16px;
    line-height:1.42;
    width:560px;
">

<div style="
    font-size:22px;
    font-weight:bold;
    color:#2f6f62;
    margin-bottom:8px;
">
Dither and Whitening of Quantization Error
</div>

<div style="margin-bottom:5px;">
Without dither, the quantization error ν[n] = Q(x[n]) − x[n] may exhibit deterministic structure and statistical dependence on the input.
</div>

<div style="margin-bottom:5px;">
Dither adds an independent random sequence d[n] before quantization, so the quantizer input becomes x[n] = y[n] + d[n].
</div>

<div style="margin-bottom:5px;">
For suitable uniform dither, the quantization error approaches a uniform, zero-mean, white-noise process with variance Δ²/12.
</div>

<div>
<b>This notebook:</b> compares the error distribution, autocorrelation and input-error dependence with and without dithering.
</div>

</div>
""")

# ============================================================
# MODE SELECTOR
# ============================================================

mode_selector = RadioButtons(
    options=['Without dither', 'With dither'],
    value='Without dither',
    description='',
    layout=Layout(width='270px')
)

mode_selector.add_class('dither-radio')

# ============================================================
# SLIDERS
# ============================================================

slider_style = {'description_width': '0px'}

slider_layout = Layout(
    width='115px',
    min_width='115px'
)

delta_slider = FloatSlider(
    min=0.20,
    max=2.00,
    step=0.10,
    value=1.00,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

amplitude_slider = FloatSlider(
    min=0.50,
    max=4.00,
    step=0.25,
    value=2.50,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

frequency_slider = FloatSlider(
    min=0.01,
    max=0.15,
    step=0.01,
    value=0.05,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

dither_span_slider = IntSlider(
    min=1,
    max=4,
    step=1,
    value=1,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    disabled=True
)

N_slider = IntSlider(
    min=500,
    max=5000,
    step=500,
    value=3000,
    description=' ',
    readout=False,
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)

# ============================================================
# VALUE LABELS
# ============================================================

value_layout = Layout(
    width='50px',
    min_width='50px',
    margin='0px 0px 0px 3px'
)

delta_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">1.00</div>',
    layout=value_layout
)

amplitude_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">2.50</div>',
    layout=value_layout
)

frequency_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">0.05</div>',
    layout=value_layout
)

dither_span_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">1</div>',
    layout=value_layout
)

N_value = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">3000</div>',
    layout=value_layout
)

# ============================================================
# LABELS
# ============================================================

label_layout = Layout(
    width='145px',
    min_width='145px'
)

mode_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Mode:</div>',
    layout=label_layout
)

delta_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Quantization step Δ:</div>',
    layout=label_layout
)

amplitude_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Signal amplitude A:</div>',
    layout=label_layout
)

frequency_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Normalized freq. f₀:</div>',
    layout=label_layout
)

dither_span_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Dither span MΔ:</div>',
    layout=label_layout
)

N_label = HTML(
    '<div style="font-family:Arial; font-size:14px; font-weight:bold; white-space:nowrap;">Samples N:</div>',
    layout=label_layout
)

# ============================================================
# CONTROL ROWS
# ============================================================

row_layout = Layout(
    width='400px',
    min_width='400px',
    height='36px',
    min_height='36px',
    align_items='center',
    overflow='visible'
)

mode_row = HBox(
    [mode_label, mode_selector],
    layout=row_layout
)

delta_row = HBox(
    [delta_label, delta_slider, delta_value],
    layout=row_layout
)

amplitude_row = HBox(
    [amplitude_label, amplitude_slider, amplitude_value],
    layout=row_layout
)

frequency_row = HBox(
    [frequency_label, frequency_slider, frequency_value],
    layout=row_layout
)

dither_span_row = HBox(
    [dither_span_label, dither_span_slider, dither_span_value],
    layout=row_layout
)

N_row = HBox(
    [N_label, N_slider, N_value],
    layout=row_layout
)

# ============================================================
# CONTROLS CARD
# ============================================================

controls_card = VBox(
    [
        HTML("""
        <div style="
            font-family:Arial;
            font-size:17px;
            font-weight:bold;
            color:#2f6f62;
            margin-bottom:8px;
        ">
        Dither Parameters
        </div>
        """),
        mode_row,
        delta_row,
        amplitude_row,
        frequency_row,
        dither_span_row,
        N_row
    ],
    layout=Layout(
        width='430px',
        min_width='430px',
        padding='10px 12px 12px 12px',
        border='1px solid #b8d7ce',
        overflow='visible'
    )
)

# ============================================================
# TOP TWO-COLUMN LAYOUT
# ============================================================

top_layout = HBox(
    [
        documentation,
        controls_card
    ],
    layout=Layout(
        width='1020px',
        align_items='flex-start',
        justify_content='space-between',
        gap='16px',
        margin='0px 0px 10px 0px',
        overflow='visible'
    )
)

# ============================================================
# OUTPUT AREAS
# ============================================================

graph_output = Output(
    layout=Layout(
        width='1080px',
        overflow='hidden'
    )
)

result_html = HTML()

# ============================================================
# MID-TREAD QUANTIZER
# ============================================================

def quantize(x, delta):
    return delta * np.floor(x / delta + 0.5)

# ============================================================
# NORMALIZED AUTOCORRELATION
# ============================================================

def normalized_acf(x, max_lag):

    x = x - np.mean(x)
    N = len(x)
    variance = np.mean(x**2)
    acf = np.zeros(max_lag + 1)

    if variance == 0:
        return acf

    for lag in range(max_lag + 1):
        acf[lag] = np.mean(x[:N - lag] * x[lag:]) / variance

    return acf

# ============================================================
# MAIN PLOT FUNCTION
# ============================================================

def plot_dither_demo(mode, delta, amplitude, frequency, dither_span, N):

    # --------------------------------------------------------
    # FIXED RANDOM GENERATOR
    # --------------------------------------------------------

    rng = np.random.default_rng(1107)

    # --------------------------------------------------------
    # ORIGINAL SIGNAL y[n]
    # --------------------------------------------------------

    n = np.arange(N)
    y = amplitude * np.sin(2.0 * np.pi * frequency * n)

    # --------------------------------------------------------
    # DITHER
    # --------------------------------------------------------

    if mode == 'With dither':

        dither_width = dither_span * delta
        d = rng.uniform(-dither_width / 2.0, dither_width / 2.0, N)

    else:

        dither_width = 0.0
        d = np.zeros(N)

    # --------------------------------------------------------
    # QUANTIZER INPUT
    # --------------------------------------------------------

    x = y + d

    # --------------------------------------------------------
    # QUANTIZED OUTPUT
    # --------------------------------------------------------

    q = quantize(x, delta)

    # --------------------------------------------------------
    # QUANTIZATION ERROR
    # --------------------------------------------------------

    nu = q - x

    # --------------------------------------------------------
    # IDEAL PQN
    # --------------------------------------------------------

    pqn = rng.uniform(-delta / 2.0, delta / 2.0, N)

    # --------------------------------------------------------
    # STATISTICS
    # --------------------------------------------------------

    mean_error = np.mean(nu)
    variance_error = np.var(nu)
    ideal_variance = delta**2 / 12.0

    # --------------------------------------------------------
    # ORIGINAL SIGNAL / ERROR CORRELATION
    # --------------------------------------------------------

    if np.std(y) > 0 and np.std(nu) > 0:
        rho_y_nu = np.corrcoef(y, nu)[0, 1]
    else:
        rho_y_nu = 0.0

    # --------------------------------------------------------
    # QUANTIZER INPUT / ERROR CORRELATION
    # --------------------------------------------------------

    if np.std(x) > 0 and np.std(nu) > 0:
        rho_x_nu = np.corrcoef(x, nu)[0, 1]
    else:
        rho_x_nu = 0.0

    # --------------------------------------------------------
    # ERROR AUTOCORRELATION
    # --------------------------------------------------------

    max_lag = 35
    lags = np.arange(max_lag + 1)

    acf_error = normalized_acf(nu, max_lag)
    acf_pqn = normalized_acf(pqn, max_lag)

    # --------------------------------------------------------
    # DISPLAY RANGE
    # --------------------------------------------------------

    display_N = min(220, N)

    # ========================================================
    # FIGURE
    # ========================================================

    fig = plt.figure(figsize=(10.6, 7.3))

    gs = fig.add_gridspec(
        2,
        2,
        hspace=0.46,
        wspace=0.30
    )

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[1, 0])
    ax4 = fig.add_subplot(gs[1, 1])

    # ========================================================
    # GRAPH 1: SIGNAL AND QUANTIZED OUTPUT
    # ========================================================

    ax1.plot(
        n[:display_N],
        y[:display_N],
        linewidth=1.3,
        label='Original y[n]'
    )

    ax1.step(
        n[:display_N],
        q[:display_N],
        where='mid',
        linewidth=1.2,
        label='Quantized output'
    )

    y_limit = max(4.0, 1.5 * amplitude + dither_width)

    ax1.set_xlim(0, display_N - 1)
    ax1.set_ylim(-y_limit, y_limit)
    ax1.set_xlabel('Sample index n', fontsize=10)
    ax1.set_ylabel('Amplitude', fontsize=10)
    ax1.set_title('Signal and Quantized Output', fontsize=12, pad=8)
    ax1.tick_params(axis='both', labelsize=9)
    ax1.grid(True, linestyle=':', alpha=0.4)
    ax1.legend(loc='upper right', fontsize=8)

    # ========================================================
    # GRAPH 2: QUANTIZATION ERROR
    # ========================================================

    ax2.plot(
        n[:display_N],
        nu[:display_N],
        linewidth=1.1
    )

    ax2.axhline(
        delta / 2.0,
        linestyle='--',
        linewidth=1.0,
        label='+Δ/2'
    )

    ax2.axhline(
        -delta / 2.0,
        linestyle='--',
        linewidth=1.0,
        label='−Δ/2'
    )

    ax2.axhline(0, linewidth=0.8)

    ax2.set_xlim(0, display_N - 1)
    ax2.set_ylim(-1.05, 1.05)
    ax2.set_xlabel('Sample index n', fontsize=10)
    ax2.set_ylabel('ν[n]', fontsize=10)
    ax2.set_title('Quantization Error', fontsize=12, pad=8)
    ax2.tick_params(axis='both', labelsize=9)
    ax2.grid(True, linestyle=':', alpha=0.4)
    ax2.legend(loc='upper right', fontsize=8)

    # ========================================================
    # GRAPH 3: ERROR PDF
    # ========================================================

    bins = np.linspace(-delta / 2.0, delta / 2.0, 35)

    ax3.hist(
        nu,
        bins=bins,
        density=True,
        alpha=0.60,
        label='Actual error'
    )

    ideal_x = np.array(
        [
            -delta / 2.0,
            -delta / 2.0,
            delta / 2.0,
            delta / 2.0
        ]
    )

    ideal_y = np.array(
        [
            0.0,
            1.0 / delta,
            1.0 / delta,
            0.0
        ]
    )

    ax3.plot(
        ideal_x,
        ideal_y,
        linewidth=2.0,
        label='Ideal uniform PDF'
    )

    ax3.set_xlim(-1.05, 1.05)
    ax3.set_ylim(0, 5.5)
    ax3.set_xlabel('Error amplitude', fontsize=10)
    ax3.set_ylabel('Probability density', fontsize=10)
    ax3.set_title('Error PDF and Uniform PQN Model', fontsize=12, pad=8)
    ax3.tick_params(axis='both', labelsize=9)
    ax3.grid(True, linestyle=':', alpha=0.4)
    ax3.legend(loc='upper right', fontsize=8)

    # ========================================================
    # GRAPH 4: AUTOCORRELATION
    # ========================================================

    ax4.plot(
        lags,
        acf_error,
        marker='o',
        markersize=3,
        linewidth=1.5,
        label='Quantization error'
    )

    ax4.plot(
        lags,
        acf_pqn,
        marker='s',
        markersize=3,
        linewidth=1.3,
        label='Ideal white PQN'
    )

    ax4.axhline(0, linewidth=0.8)

    ax4.set_xlim(0, max_lag)
    ax4.set_ylim(-1.05, 1.05)
    ax4.set_xlabel('Lag k', fontsize=10)
    ax4.set_ylabel('Normalized autocorrelation', fontsize=10)
    ax4.set_title('Whitening of the Error', fontsize=12, pad=8)
    ax4.tick_params(axis='both', labelsize=9)
    ax4.grid(True, linestyle=':', alpha=0.4)
    ax4.legend(loc='upper right', fontsize=8)

    # ========================================================
    # FIGURE SPACING
    # ========================================================

    fig.subplots_adjust(
        left=0.08,
        right=0.97,
        top=0.93,
        bottom=0.09
    )

    plt.show()
    plt.close(fig)

    # ========================================================
    # MODE TEXT
    # ========================================================

    if mode == 'With dither':
        mode_text = f'Uniform dither: d[n] ∈ [−{dither_span}Δ/2, +{dither_span}Δ/2]'
    else:
        mode_text = 'No dither is added before quantization.'

    # ========================================================
    # RESULT BOX
    # ========================================================

    result_html.value = f"""
    <div style="
        font-family:Arial, sans-serif;
        font-size:15px;
        line-height:1.50;
        width:980px;
        padding:10px 14px;
        border:1px solid #d7c38d;
        background:#fffbed;
        box-sizing:border-box;
    ">

    <b>Current mode:</b>
    {mode_text}

    <br>

    <b>Actual quantization error:</b>
    E{{ν}} = {mean_error:.5f}

    &nbsp;&nbsp;&nbsp;

    Var{{ν}} = {variance_error:.5f}

    <br>

    <b>Ideal uniform-noise variance:</b>
    Δ²/12 = {ideal_variance:.5f}

    <br>

    <b>Original signal / error correlation:</b>
    ρ(y,ν) = {rho_y_nu:.5f}

    &nbsp;&nbsp;&nbsp;

    <b>Quantizer input / error correlation:</b>
    ρ(x,ν) = {rho_x_nu:.5f}

    </div>
    """

# ============================================================
# UPDATE FUNCTION
# ============================================================

def update_notebook(change=None):

    dither_span_slider.disabled = (
        mode_selector.value == 'Without dither'
    )

    delta_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{delta_slider.value:.2f}</div>'
    amplitude_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{amplitude_slider.value:.2f}</div>'
    frequency_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{frequency_slider.value:.2f}</div>'
    dither_span_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{dither_span_slider.value}</div>'
    N_value.value = f'<div style="font-family:Arial; font-size:14px; font-weight:bold; color:#0b3d91;">{N_slider.value}</div>'

    with graph_output:

        clear_output(wait=True)

        plot_dither_demo(
            mode_selector.value,
            delta_slider.value,
            amplitude_slider.value,
            frequency_slider.value,
            dither_span_slider.value,
            N_slider.value
        )

# ============================================================
# CONNECT CONTROLS
# ============================================================

mode_selector.observe(update_notebook, names='value')
delta_slider.observe(update_notebook, names='value')
amplitude_slider.observe(update_notebook, names='value')
frequency_slider.observe(update_notebook, names='value')
dither_span_slider.observe(update_notebook, names='value')
N_slider.observe(update_notebook, names='value')

# ============================================================
# INTERPRETATION
# ============================================================

interpretation = HTML("""
<div style="
    font-family:Arial, sans-serif;
    font-size:15px;
    line-height:1.45;
    width:1080px;
    padding:11px 15px;
    border:1px solid #bdd7d0;
    background:#f7fcfa;
    box-sizing:border-box;
    margin-top:6px;
">

<div style="
    font-size:18px;
    font-weight:bold;
    color:#2f6f62;
    margin-bottom:6px;
">
Interpretation of the Results
</div>

<div style="margin-bottom:4px;">
Without dither, a periodic input can produce a structured quantization error whose samples are correlated in time and may depend statistically on the signal.
</div>

<div style="margin-bottom:4px;">
Adding independent uniform dither randomizes the quantizer operating point and makes the error distribution approach the ideal uniform model with variance Δ²/12.
</div>

<div>
The corresponding autocorrelation becomes increasingly concentrated at zero lag, illustrating why dithering can make quantization error behave more like white noise.
</div>

</div>
""")

# ============================================================
# INITIAL DRAW
# ============================================================

update_notebook()

# ============================================================
# COMPLETE NOTEBOOK
# ============================================================

main_layout = VBox(
    [
        top_layout,
        graph_output,
        result_html,
        interpretation
    ],
    layout=Layout(
        width='1080px',
        overflow='visible'
    )
)

display(main_layout)